# PPC v10 — Unified Single-Notebook Training (v8 architecture + v9 GAN refiner)

**What this notebook does (single file, train from scratch):**

- **Phase 1 (Cell 8):** Trains `PPCNetV8` from scratch on 829 training patients. Architecture is identical to v8 (the proven 2.210 mm Chamfer baseline). Projection matrices `P_ap` / `P_lp` are used both for 2D feature sampling at projected query points *and* for the 2D-Chamfer projection-consistency loss, exactly as in v8.
- **Phase 2 (Cell 9, optional):** After Phase 1 converges, freezes the generator and trains a SN-GAN refiner on top of it (PointRefiner + spectral-normalized PointNet discriminator), exactly as in v9.
- **Cell 10:** Test-set evaluation with VTK saving.
- **Cell 11:** `predict_ppc()` inference function.

**Settings preserved from working v8/v9:**
- 50 GB VRAM cap (50% of A100-80GB) — shared cluster-friendly.
- `NUM_WORKERS=4` with `persistent_workers=True` (proven not to kill the kernel in v8).
- `BATCH_SIZE=2`, AMP enabled, cosine LR with 10-epoch warmup.
- Gap-preserving losses with curriculum (epochs 15→40), exactly as v8.
- Checkpoint resume from `latest_checkpoint.pth`.

**Project directory:** `/apps/app/pandu/ppc_network_v10`

Run cells 1→8 to train the generator from scratch. Run Cell 9 only after Cell 8 finishes (or after restoring a v10 generator checkpoint). Cell 10 evaluates whichever stage is available.

In [3]:
"""
PPC v10 — Cell 1: Imports, paths, hyper-parameters, VRAM cap.
"""
import os, sys, json, time, math, random, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import vtk

warnings.filterwarnings('ignore', category=FutureWarning)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Total VRAM: {total_gb:.1f} GB')
    # 50% VRAM cap — shared cluster friendly
    fraction = min(50.0 / total_gb, 0.95)
    torch.cuda.set_per_process_memory_fraction(fraction, device=0)
    print(f'VRAM cap   : {fraction*100:.1f}% (~{fraction*total_gb:.1f} GB)')

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

# ── PATHS ─────────────────────────────────────────────────────────────────────
DATA_ROOT   = Path('/apps/app/see_all_ai/dataset/Lumbar_Filtered_1037')
SPLIT_FILE  = DATA_ROOT / 'dataset_split.json'
PROJECT_DIR = Path('/apps/app/see_all_ai/ppc_network_v10')
CKPT_DIR    = PROJECT_DIR / 'checkpoints'
LOG_DIR     = PROJECT_DIR / 'logs'
RESULTS_DIR = PROJECT_DIR / 'results'
for d in [PROJECT_DIR, CKPT_DIR, LOG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── MODEL CONFIG (identical to v8) ───────────────────────────────────────────
IMG_SIZE         = 512
COARSE_VOL_SIZE  = 32
AUX_VOL_SIZE     = 48
GAP_VOL_SIZE     = 96      # high-res, NO dilation → gaps stay empty
N_POINTS         = 5120
ENC_CHANNELS     = 192
VOL_CHANNELS     = 128
DEC_CHANNELS     = 128
QUERY_DIM        = 256
N_REFINE_STAGES  = 3
PRETRAINED       = True

# ── PHASE 1 (generator) TRAINING ──────────────────────────────────────────────
BATCH_SIZE       = 2
NUM_WORKERS      = 4
EPOCHS           = 250
LR               = 8e-5
WEIGHT_DECAY     = 1e-5
WARMUP_EPOCHS    = 10
USE_AMP          = True
GRAD_CLIP        = 1.0

# ── PHASE 1 LOSS WEIGHTS (v8 gap-preserving) ──────────────────────────────────
W_CHAMFER   = 1.00   # asymmetric chamfer (GT→pred weighted 1.5×)
W_GAP       = 0.50   # empty-voxel penalty — the gap-preserving term
W_AXIAL     = 0.40   # soft Z-density matching
W_SW        = 0.25   # sliced Wasserstein
W_REPULSION = 0.02   # anisotropic (Z-favored)
W_AUX_OCC   = 0.05   # coarse occupancy supervision
W_OFFSET    = 0.0005
W_PROJ      = 0.02   # 2D Chamfer projection-consistency (uses P_ap, P_lp)

# ── PHASE 2 (GAN refiner) TRAINING ────────────────────────────────────────────
GAN_EPOCHS      = 150
LR_G            = 2e-5     # refiner LR
LR_D            = 1e-4     # discriminator LR
N_CRITIC        = 3        # D updates per G update
D_SUBSAMPLE     = 1024     # subsample points for discriminator (memory)
W_ADV           = 0.10
W_CHAMFER_R     = 1.00
W_GAP_R         = 0.60
W_AXIAL_R       = 0.50
W_SW_R          = 0.30

MAX_EVAL_SAMPLES = 103

# ── CHECKPOINT PATHS ──────────────────────────────────────────────────────────
# Phase 1 (generator)
GEN_CKPT_PATH      = CKPT_DIR / 'gen_latest.pth'
GEN_BEST_CKPT      = CKPT_DIR / 'gen_best.pth'
GEN_BEST_CH_CKPT   = CKPT_DIR / 'gen_best_chamfer.pth'
GEN_LOG            = LOG_DIR  / 'gen_training_log.json'

# Phase 2 (GAN refiner)
GAN_CKPT_PATH      = CKPT_DIR / 'gan_latest.pth'
GAN_BEST_CKPT      = CKPT_DIR / 'gan_best_chamfer.pth'
GAN_LOG            = LOG_DIR  / 'gan_training_log.json'

print('=' * 72)
print('CONFIGURATION — PPC v10 (Unified v8 generator + v9 GAN refiner)')
print('=' * 72)
cfg = dict(IMG_SIZE=IMG_SIZE, N_POINTS=N_POINTS, GAP_VOL_SIZE=GAP_VOL_SIZE,
           BATCH_SIZE=BATCH_SIZE, EPOCHS=EPOCHS, LR=LR,
           W_CHAMFER=W_CHAMFER, W_GAP=W_GAP, W_AXIAL=W_AXIAL, W_SW=W_SW,
           W_PROJ=W_PROJ, N_REFINE_STAGES=N_REFINE_STAGES,
           GAN_EPOCHS=GAN_EPOCHS, W_ADV=W_ADV, D_SUBSAMPLE=D_SUBSAMPLE)
for k, v in cfg.items():
    print(f'  {k:<20} = {v}')
print(f'  PROJECT_DIR        = {PROJECT_DIR}')


/apps/app/see_all_ai/seeallai/lib/python3.10/site-packages/_distutils_hack/__init__.py:53: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


Device: cuda
GPU: NVIDIA A100-SXM4-80GB
Total VRAM: 85.0 GB
VRAM cap   : 58.8% (~50.0 GB)
CONFIGURATION — PPC v10 (Unified v8 generator + v9 GAN refiner)
  IMG_SIZE             = 512
  N_POINTS             = 5120
  GAP_VOL_SIZE         = 96
  BATCH_SIZE           = 2
  EPOCHS               = 250
  LR                   = 8e-05
  W_CHAMFER            = 1.0
  W_GAP                = 0.5
  W_AXIAL              = 0.4
  W_SW                 = 0.25
  W_PROJ               = 0.02
  N_REFINE_STAGES      = 3
  GAN_EPOCHS           = 150
  W_ADV                = 0.1
  D_SUBSAMPLE          = 1024
  PROJECT_DIR        = /apps/app/see_all_ai/ppc_network_v10


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — DATA UTILITIES (VTK I/O, DRR loading, projection matrices, occupancy)
# ══════════════════════════════════════════════════════════════════════════════

def load_vtk_points(path):
    reader = vtk.vtkPolyDataReader()
    reader.SetFileName(str(path))
    reader.Update()
    poly = reader.GetOutput()
    n = poly.GetNumberOfPoints()
    return np.array([poly.GetPoint(i) for i in range(n)], dtype=np.float32)


def save_vtk_points(points, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    vp = vtk.vtkPoints()
    for p in points:
        vp.InsertNextPoint(float(p[0]), float(p[1]), float(p[2]))
    vc = vtk.vtkCellArray()
    for i in range(len(points)):
        vc.InsertNextCell(1)
        vc.InsertCellPoint(i)
    poly = vtk.vtkPolyData()
    poly.SetPoints(vp)
    poly.SetVerts(vc)
    w = vtk.vtkPolyDataWriter()
    w.SetFileName(str(output_path))
    w.SetInputData(poly)
    w.SetFileTypeToASCII()
    w.Write()
    return output_path


def load_drr_image(path, size=IMG_SIZE):
    img = Image.open(path).convert('L')
    if img.size != (size, size):
        img = img.resize((size, size), Image.BILINEAR)
    return np.array(img, dtype=np.float32) / 255.0


def load_projection_matrix(path):
    """Load a 3×4 projection matrix from a whitespace-separated text file."""
    with open(path, 'r') as f:
        vals = [float(v) for v in f.read().split()]
    return np.array(vals, dtype=np.float32).reshape(3, 4)


def load_split(split_path=SPLIT_FILE):
    with open(split_path, 'r') as f:
        d = json.load(f)
    return d['train'], d['val'], d['test']


def append_training_log(log_path, epoch_record):
    log = {'records': []}
    if log_path.exists():
        with open(log_path, 'r') as f:
            log = json.load(f)
    log['records'].append(epoch_record)
    tmp = log_path.with_suffix('.json.tmp')
    with open(tmp, 'w') as f:
        json.dump(log, f, indent=2)
    tmp.replace(log_path)


def points_to_local(points, center, scale):
    return (points - center[None]) / (scale[None] + 1e-6)


def local_to_world(points_local, center, scale):
    return points_local * scale[:, None, :] + center[:, None, :]


def compute_scale(gt_full):
    mins = gt_full.min(axis=0)
    maxs = gt_full.max(axis=0)
    extent = (maxs - mins).astype(np.float32)
    scale = extent * 0.55 + np.array([20., 20., 30.], dtype=np.float32)
    return np.maximum(scale, np.array([50., 50., 80.], dtype=np.float32))


def make_aux_occupancy(points_local, vol_size=AUX_VOL_SIZE, dilate=1):
    """Coarse occupancy for U-Net auxiliary supervision (dilated = filled)."""
    pts = np.clip((np.asarray(points_local, np.float32) + 1.0) * 0.5, 0.0, 0.999999)
    idx = np.clip(np.floor(pts * vol_size).astype(np.int32), 0, vol_size - 1)
    occ = np.zeros((vol_size, vol_size, vol_size), dtype=np.float32)
    for dx in range(-dilate, dilate + 1):
        for dy in range(-dilate, dilate + 1):
            for dz in range(-dilate, dilate + 1):
                x = np.clip(idx[:, 0] + dx, 0, vol_size - 1)
                y = np.clip(idx[:, 1] + dy, 0, vol_size - 1)
                z = np.clip(idx[:, 2] + dz, 0, vol_size - 1)
                occ[x, y, z] = 1.0
    return occ


def make_gap_occupancy(points_local, vol_size=GAP_VOL_SIZE, dilate=0):
    """HIGH-RES occupancy with NO dilation — gaps stay empty.
    At 96³ with typical scale, Z-voxel ≈ 2.5 mm → resolves 3–5 mm gaps.
    """
    pts = np.clip((np.asarray(points_local, np.float32) + 1.0) * 0.5, 0.0, 0.999999)
    idx = np.clip(np.floor(pts * vol_size).astype(np.int32), 0, vol_size - 1)
    occ = np.zeros((vol_size, vol_size, vol_size), dtype=np.float32)
    occ[idx[:, 0], idx[:, 1], idx[:, 2]] = 1.0
    return occ


def flip_projection_matrix_horizontal(P, img_size=IMG_SIZE):
    """Mirror-flip a projection matrix to match a horizontally-flipped image."""
    F_mat = np.array([
        [-1, 0, img_size - 1],
        [ 0, 1, 0           ],
        [ 0, 0, 1           ],
    ], dtype=np.float32)
    return F_mat @ P


AUG_INTENSITY = 0.15
AUG_FLIP_PROB = 0.5

print('Data utilities ready (VTK I/O, DRR, projection matrices, occupancy grids).')


Data utilities ready (VTK I/O, DRR, projection matrices, occupancy grids).


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — DATASET
# ══════════════════════════════════════════════════════════════════════════════

class LumbarDatasetV10(Dataset):
    """Loads AP/LP DRRs, projection matrices, GT point cloud, and both
    occupancy grids (aux_occ dilated for U-Net, gap_occ at 96³ NOT dilated
    for the gap-penalty loss).
    """
    def __init__(self, patient_ids, data_root=DATA_ROOT, img_size=IMG_SIZE, augment=False):
        self.patient_ids = patient_ids
        self.data_root   = Path(data_root)
        self.img_size    = img_size
        self.augment     = augment
        self.img_norm    = transforms.Normalize(mean=[0.485], std=[0.229])

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        pid  = self.patient_ids[idx]
        pdir = self.data_root / pid

        drr_ap = load_drr_image(pdir / 'AP_0'  / 'drr_AP_0.png',  self.img_size)
        drr_lp = load_drr_image(pdir / 'LP_90' / 'drr_LP_90.png', self.img_size)
        P_ap   = load_projection_matrix(pdir / 'AP_0'  / 'P_AP_0.txt')
        P_lp   = load_projection_matrix(pdir / 'LP_90' / 'P_LP_90.txt')
        gt_full = load_vtk_points(pdir / 'gt_ppc.vtk')

        center = gt_full.mean(axis=0).astype(np.float32)
        scale  = compute_scale(gt_full)
        gt_local_full = np.clip(points_to_local(gt_full, center, scale), -1.0, 1.0)

        aux_occ = make_aux_occupancy(gt_local_full, AUX_VOL_SIZE, dilate=1)
        gap_occ = make_gap_occupancy(gt_local_full, GAP_VOL_SIZE, dilate=0)

        n = len(gt_full)
        sel = np.random.choice(n, N_POINTS, replace=(n < N_POINTS))
        gt_world = gt_full[sel].astype(np.float32)
        gt_local = gt_local_full[sel].astype(np.float32)

        if self.augment:
            for drr in [drr_ap, drr_lp]:
                jitter = 1.0 + (np.random.rand() * 2 - 1) * AUG_INTENSITY
                drr[:] = np.clip(drr * jitter, 0.0, 1.0)
            if np.random.rand() < AUG_FLIP_PROB:
                drr_ap = drr_ap[:, ::-1].copy()
                drr_lp = drr_lp[:, ::-1].copy()
                P_ap = flip_projection_matrix_horizontal(P_ap, self.img_size)
                P_lp = flip_projection_matrix_horizontal(P_lp, self.img_size)

        drr_ap_t = self.img_norm(torch.from_numpy(drr_ap).unsqueeze(0).float())
        drr_lp_t = self.img_norm(torch.from_numpy(drr_lp).unsqueeze(0).float())

        return {
            'drr_ap':       drr_ap_t,
            'drr_lp':       drr_lp_t,
            'P_ap':         torch.from_numpy(P_ap).float(),
            'P_lp':         torch.from_numpy(P_lp).float(),
            'gt_ppc_world': torch.from_numpy(gt_world).float(),
            'gt_ppc_local': torch.from_numpy(gt_local).float(),
            'aux_occ':      torch.from_numpy(aux_occ).float(),
            'gap_occ':      torch.from_numpy(gap_occ).float(),
            'center':       torch.from_numpy(center).float(),
            'scale':        torch.from_numpy(scale).float(),
            'patient_id':   pid,
        }


print('LumbarDatasetV10 defined.')


LumbarDatasetV10 defined.


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — LOAD TRAIN / VAL / TEST SPLIT
# ══════════════════════════════════════════════════════════════════════════════
train_ids, val_ids, test_ids = load_split()
print(f'Split: train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}')


Split: train=829  val=103  test=105


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — MODEL ARCHITECTURE (identical to PPCNetV8 — proven 2.210 mm baseline)
#   Uses projection matrices P_ap / P_lp in TWO ways:
#     (a) project_points() during refinement — to sample 2D features at the
#         projected (u,v) location of each 3D query point.
#     (b) project_consistency_loss in Cell 6 — 2D Chamfer in image space.
# ══════════════════════════════════════════════════════════════════════════════

class Encoder2D(nn.Module):
    """ResNet-18 backbone returning a 192-channel feature map."""
    def __init__(self, in_channels=1, out_channels=ENC_CHANNELS, pretrained=PRETRAINED):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        new_c1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        if pretrained:
            with torch.no_grad():
                new_c1.weight[:] = base.conv1.weight.mean(dim=1, keepdim=True)
        base.conv1 = new_c1
        self.stem   = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.proj   = nn.Conv2d(256, out_channels, kernel_size=1)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.proj(x)


class FeatureLift(nn.Module):
    """Lift 2D feature map to 3D volume via depth embedding + 3D refinement."""
    def __init__(self, in_channels=ENC_CHANNELS, out_channels=VOL_CHANNELS, depth=COARSE_VOL_SIZE):
        super().__init__()
        self.depth_embed = nn.Parameter(torch.randn(1, in_channels, depth, 1, 1) * 0.02)
        self.refine = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels), nn.GELU(),
            nn.Conv3d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels), nn.GELU(),
        )

    def forward(self, feat_2d):
        B, C, H, W = feat_2d.shape
        vol = feat_2d.unsqueeze(2).expand(B, C, COARSE_VOL_SIZE, H, W).contiguous()
        vol = vol + self.depth_embed
        return self.refine(vol)


class BiplanarFusion(nn.Module):
    """Align AP and LP volumes to a common 3D frame and fuse."""
    def __init__(self, in_channels=VOL_CHANNELS, out_channels=VOL_CHANNELS):
        super().__init__()
        self.fuse = nn.Sequential(
            nn.Conv3d(in_channels * 2, out_channels, 1),
            nn.GroupNorm(8, out_channels), nn.GELU(),
            nn.Conv3d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels), nn.GELU(),
        )

    def forward(self, ap_vol, lp_vol):
        ap_aligned = ap_vol.permute(0, 1, 4, 2, 3).contiguous()
        lp_aligned = lp_vol.permute(0, 1, 2, 4, 3).contiguous()
        return self.fuse(torch.cat([ap_aligned, lp_aligned], dim=1))


class Block3D(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, padding=1), nn.GroupNorm(8, out_c), nn.GELU(),
            nn.Conv3d(out_c, out_c, 3, padding=1), nn.GroupNorm(8, out_c), nn.GELU(),
        )
    def forward(self, x): return self.block(x)


class CoarseUNet3D(nn.Module):
    """3D U-Net producing per-voxel features + auxiliary occupancy logits."""
    def __init__(self, in_channels=VOL_CHANNELS, feat_channels=DEC_CHANNELS):
        super().__init__()
        self.enc1       = Block3D(in_channels, 96)
        self.down1      = nn.Conv3d(96,  160, 2, stride=2)
        self.enc2       = Block3D(160, 160)
        self.down2      = nn.Conv3d(160, 224, 2, stride=2)
        self.bottleneck = Block3D(224, 224)
        self.up2        = nn.ConvTranspose3d(224, 160, 2, stride=2)
        self.dec2       = Block3D(320, 160)
        self.up1        = nn.ConvTranspose3d(160, 96,  2, stride=2)
        self.dec1       = Block3D(192, feat_channels)
        self.aux_head   = nn.Sequential(
            nn.Conv3d(feat_channels, feat_channels // 2, 3, padding=1),
            nn.GELU(),
            nn.Conv3d(feat_channels // 2, 1, 1),
        )

    def forward(self, x):
        e1  = self.enc1(x)
        e2  = self.enc2(self.down1(e1))
        b   = self.bottleneck(self.down2(e2))
        d2  = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1  = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        aux = F.interpolate(self.aux_head(d1),
                            size=(AUX_VOL_SIZE,) * 3,
                            mode='trilinear', align_corners=True)
        return d1, aux


# ── PROJECTION HELPERS (use P_ap / P_lp explicitly) ──────────────────────────
def project_points(P, points_world, img_size=IMG_SIZE):
    """Project a batch of 3D points (B,N,3) through the 3×4 projection matrix
    P to 2D pixel coordinates uv and to grid_sample-normalized [-1,1] coords.
    Used inside every refinement stage to look up 2D features per query.
    """
    B, N, _ = points_world.shape
    ones  = torch.ones(B, N, 1, device=points_world.device, dtype=points_world.dtype)
    homog = torch.cat([points_world, ones], dim=-1)
    uvw   = torch.bmm(homog, P.transpose(1, 2))
    z     = uvw[..., 2:3].clamp(min=1e-6)
    uv    = uvw[..., :2] / z
    uv_norm = (uv / (img_size - 1.0)) * 2.0 - 1.0
    return uv, uv_norm, torch.log(z)


def sample_2d_features(feat_map, uv_norm):
    """Bilinear sample 2D features at the projected (u,v) location."""
    grid = uv_norm.view(feat_map.shape[0], -1, 1, 2)
    samp = F.grid_sample(feat_map, grid, mode='bilinear',
                         align_corners=True, padding_mode='border')
    return samp.squeeze(-1).transpose(1, 2)


def sample_3d_features(vol_feat, points_local):
    """Bilinear sample 3D features at the query's local (x,y,z) in [-1,1]."""
    grid = torch.stack([points_local[..., 2],
                        points_local[..., 1],
                        points_local[..., 0]], dim=-1)
    grid = grid.view(points_local.shape[0], -1, 1, 1, 3)
    samp = F.grid_sample(vol_feat, grid, mode='bilinear',
                         align_corners=True, padding_mode='border')
    return samp.squeeze(-1).squeeze(-1).transpose(1, 2)


# ── QUERY INITIALIZER (occupancy-gated offsets) ──────────────────────────────
class QueryInitializerV8(nn.Module):
    """Predicts initial 3D query positions from the bottleneck volume feature.
    Predicted occupancy gates the per-point offsets: points in empty regions
    keep smaller offsets so the gap penalty can push them toward bone.
    """
    def __init__(self, n_points=N_POINTS, feat_channels=DEC_CHANNELS):
        super().__init__()
        xs = np.linspace(-0.8, 0.8, 16)
        ys = np.linspace(-0.8, 0.8, 16)
        zs = np.linspace(-0.9, 0.9, 20)
        grid = np.stack(np.meshgrid(xs, ys, zs, indexing='ij'), -1).reshape(-1, 3).astype(np.float32)
        self.register_buffer('base_queries', torch.from_numpy(grid))
        self.n_points = n_points
        self.global_head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1), nn.Flatten(),
            nn.Linear(feat_channels, 256), nn.GELU(),
            nn.Linear(256, n_points * 3),
        )

    def forward(self, vol_feat, aux_occ_logits=None):
        B = vol_feat.shape[0]
        offset = self.global_head(vol_feat).view(B, self.n_points, 3)
        raw_q = self.base_queries[None] + 0.20 * torch.tanh(offset)

        if aux_occ_logits is not None:
            gs = torch.stack([raw_q[..., 2], raw_q[..., 1], raw_q[..., 0]], dim=-1)
            gs = gs.view(B, self.n_points, 1, 1, 3)
            occ_vals = F.grid_sample(
                aux_occ_logits, gs, mode='bilinear',
                align_corners=True, padding_mode='border'
            ).view(B, self.n_points)
            gate = torch.sigmoid(occ_vals).unsqueeze(-1)
            raw_q = self.base_queries[None] + gate * 0.25 * torch.tanh(offset)
        return torch.clamp(raw_q, -1.0, 1.0)


# ── REFINEMENT STAGE (uses P_ap / P_lp via project_points) ───────────────────
class RefinementStageV8(nn.Module):
    """Each stage samples a fused feature vector per query: 3D volume features +
    2D AP features (at projected uv) + 2D LP features (at projected uv) + the
    current local position + projected normalized uvs + sampled occupancy.
    The MLP predicts a small ±0.25 tanh delta in local space.
    """
    def __init__(self, feat_2d=ENC_CHANNELS, feat_3d=DEC_CHANNELS, hidden=QUERY_DIM):
        super().__init__()
        in_dim = feat_2d * 2 + feat_3d + 3 + 2 + 2 + 1
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def forward(self, q_local, vol_feat, feat_ap, feat_lp, P_ap, P_lp,
                center, scale, aux_occ_logits=None):
        q_world = local_to_world(q_local, center, scale)
        _, uvn_ap, _ = project_points(P_ap, q_world)   # ← uses P_ap
        _, uvn_lp, _ = project_points(P_lp, q_world)   # ← uses P_lp
        f3d = sample_3d_features(vol_feat, q_local)
        fap = sample_2d_features(feat_ap, uvn_ap)      # ← features at projected (u,v)
        flp = sample_2d_features(feat_lp, uvn_lp)

        B, N, _ = q_local.shape
        if aux_occ_logits is not None:
            gs = torch.stack([q_local[..., 2], q_local[..., 1], q_local[..., 0]], dim=-1)
            gs = gs.view(B, N, 1, 1, 3)
            occ_val = F.grid_sample(
                aux_occ_logits, gs, mode='bilinear',
                align_corners=True, padding_mode='border'
            ).view(B, N, 1)
        else:
            occ_val = torch.zeros(B, N, 1, device=q_local.device)

        x = torch.cat([f3d, fap, flp, q_local, uvn_ap, uvn_lp, occ_val], dim=-1)
        delta = 0.25 * torch.tanh(self.mlp(x))
        return q_local + delta, {'delta': delta}


# ── FULL GENERATOR NETWORK ──────────────────────────────────────────────────
class PPCNetV10(nn.Module):
    """Identical to PPCNetV8 — kept under the v10 name for clarity."""
    def __init__(self):
        super().__init__()
        self.encoder_ap = Encoder2D()
        self.encoder_lp = Encoder2D()
        self.lift_ap    = FeatureLift()
        self.lift_lp    = FeatureLift()
        self.fusion     = BiplanarFusion()
        self.coarse3d   = CoarseUNet3D()
        self.query_init = QueryInitializerV8()
        self.stages     = nn.ModuleList([RefinementStageV8() for _ in range(N_REFINE_STAGES)])

    def forward(self, drr_ap, drr_lp, P_ap, P_lp, center, scale):
        feat_ap = self.encoder_ap(drr_ap)
        feat_lp = self.encoder_lp(drr_lp)
        vol_ap  = self.lift_ap(feat_ap)
        vol_lp  = self.lift_lp(feat_lp)
        fused            = self.fusion(vol_ap, vol_lp)
        vol_feat, aux_occ = self.coarse3d(fused)

        q_local = self.query_init(vol_feat, aux_occ)

        stage_aux = []
        for stage in self.stages:
            q_local, aux = stage(q_local, vol_feat, feat_ap, feat_lp,
                                 P_ap, P_lp, center, scale, aux_occ)
            stage_aux.append(aux)

        q_local = torch.clamp(q_local, -1.0, 1.0)
        q_world = local_to_world(q_local, center, scale)

        return {
            'pred_local':     q_local,
            'pred_world':     q_world,
            'aux_occ_logits': aux_occ,
            'stage_aux':      stage_aux,
        }


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


_test_model = PPCNetV10()
print(f'PPCNetV10 (generator) parameters: {count_params(_test_model)/1e6:.1f} M')
del _test_model


PPCNetV10 (generator) parameters: 21.8 M


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — LOSS FUNCTIONS (all differentiable, identical to v8)
#   project_consistency_loss uses P_ap / P_lp directly as a 2D-Chamfer regularizer.
# ══════════════════════════════════════════════════════════════════════════════

def pairwise_sqdist(x, y):
    x2 = (x ** 2).sum(-1, keepdim=True)
    y2 = (y ** 2).sum(-1).unsqueeze(1)
    xy = torch.bmm(x, y.transpose(1, 2))
    return (x2 + y2 - 2 * xy).clamp_min(0.0)


def asymmetric_chamfer_loss(pred, gt, w_p2g=1.0, w_g2p=1.5, trunc_mm=15.0):
    """GT→pred weighted MORE so the model covers all GT regions (incl. gaps).
    pred→GT truncated to avoid pulling gap-filling points toward bone.
    """
    d2 = pairwise_sqdist(pred, gt)
    d_p2g = torch.clamp(d2.min(dim=2)[0], max=trunc_mm**2)
    d_g2p = d2.min(dim=1)[0]
    return w_p2g * d_p2g.mean() + w_g2p * d_g2p.mean()


def gap_penalty_loss(pred_local, gap_occ_gt):
    """Penalize predicted points falling in empty GT voxels (gaps)."""
    B, N, _ = pred_local.shape
    occ = gap_occ_gt.unsqueeze(1)
    grid = torch.stack([pred_local[..., 2], pred_local[..., 1],
                        pred_local[..., 0]], dim=-1)
    grid = grid.view(B, N, 1, 1, 3)
    sampled = F.grid_sample(occ, grid, mode='bilinear',
                            align_corners=True, padding_mode='border')
    occ_at_pts = sampled.view(B, N)
    return (1.0 - occ_at_pts).clamp(min=0).mean()


def soft_axial_density_loss(pred_local, gt_local, n_bins=60, sigma=0.025):
    """Match Z-axis density via Gaussian KDE — gap valleys preserved."""
    pred_z = pred_local[..., 2]
    gt_z   = gt_local[..., 2]
    centers = torch.linspace(-1.0, 1.0, n_bins, device=pred_local.device)
    p_kde = torch.exp(-0.5 * ((pred_z.unsqueeze(-1) - centers) / sigma)**2)
    g_kde = torch.exp(-0.5 * ((gt_z.unsqueeze(-1) - centers) / sigma)**2)
    p_d = p_kde.sum(dim=1); g_d = g_kde.sum(dim=1)
    p_d = p_d / (p_d.sum(dim=1, keepdim=True) + 1e-8)
    g_d = g_d / (g_d.sum(dim=1, keepdim=True) + 1e-8)
    return (p_d - g_d).abs().sum(dim=1).mean()


def sliced_wasserstein_loss(pred, gt, n_proj=50):
    """1D Wasserstein along random 3D projections."""
    dirs = F.normalize(torch.randn(n_proj, 3, device=pred.device), dim=-1)
    p_proj = torch.einsum('bnd,pd->bnp', pred, dirs)
    g_proj = torch.einsum('bnd,pd->bnp', gt, dirs)
    return ((p_proj.sort(dim=1)[0] - g_proj.sort(dim=1)[0])**2).mean()


def repulsion_loss_aniso(points, k=8, h_xy=0.020, h_z=0.040):
    """Anisotropic repulsion: larger Z-tolerance preserves vertical gaps."""
    d2 = pairwise_sqdist(points, points)
    B, N, _ = d2.shape
    eye = torch.eye(N, device=points.device, dtype=torch.bool).unsqueeze(0)
    d2 = d2.masked_fill(eye, float('inf'))
    knn_idx = torch.topk(d2, k=k, dim=-1, largest=False).indices
    pts_exp = points.unsqueeze(2).expand(B, N, k, 3)
    nbr = torch.gather(points.unsqueeze(1).expand(B, N, N, 3), 2,
                       knn_idx.unsqueeze(-1).expand(B, N, k, 3))
    diff = pts_exp - nbr
    d2_a = (diff[..., 0]**2 + diff[..., 1]**2) / (h_xy**2) + diff[..., 2]**2 / (h_z**2)
    return torch.exp(-d2_a).mean()


def project_consistency_loss(pred_world, gt_world, P_ap, P_lp):
    """2D Chamfer between projected predictions and projected GT in image
    space. Directly uses P_ap and P_lp.
    """
    uvp_ap, _, _ = project_points(P_ap, pred_world)
    uvg_ap, _, _ = project_points(P_ap, gt_world)
    uvp_lp, _, _ = project_points(P_lp, pred_world)
    uvg_lp, _, _ = project_points(P_lp, gt_world)
    def ch(a, b):
        d2 = pairwise_sqdist(a, b)
        return 0.5 * (d2.min(2)[0].mean() + d2.min(1)[0].mean())
    return ch(uvp_ap, uvg_ap) + ch(uvp_lp, uvg_lp)


def dice_loss_from_logits(logits, targets, eps=1e-6):
    probs   = torch.sigmoid(logits).reshape(logits.shape[0], -1)
    targets = targets.reshape(targets.shape[0], -1)
    inter   = (probs * targets).sum(dim=1)
    denom   = probs.sum(dim=1) + targets.sum(dim=1)
    return 1.0 - ((2 * inter + eps) / (denom + eps)).mean()


def total_loss(output, batch, current_epoch=0):
    """v8 gap-preserving total loss with ramp on gap/axial/SW (epochs 15→40)
    and decaying projection-consistency bonus.
    """
    pred_w = output['pred_world']
    pred_l = output['pred_local']
    gt_w   = batch['gt_ppc_world']
    gt_l   = batch['gt_ppc_local']

    l_ch  = asymmetric_chamfer_loss(pred_w, gt_w)
    l_gap = gap_penalty_loss(pred_l, batch['gap_occ'])
    l_ax  = soft_axial_density_loss(pred_l, gt_l)
    l_sw  = sliced_wasserstein_loss(pred_l, gt_l)
    l_rep = repulsion_loss_aniso(pred_l)
    l_aux = (F.binary_cross_entropy_with_logits(
                output['aux_occ_logits'].squeeze(1), batch['aux_occ'])
             + dice_loss_from_logits(
                output['aux_occ_logits'].squeeze(1), batch['aux_occ']))
    l_off = torch.stack([a['delta'].abs().mean() for a in output['stage_aux']]).mean()

    w_proj_eff = W_PROJ + max(0.0, (30 - current_epoch) / 30.0) * 0.08
    l_proj = project_consistency_loss(pred_w, gt_w, batch['P_ap'], batch['P_lp'])

    ramp = min(1.0, max(0.0, (current_epoch - 15) / 25.0))

    loss = (W_CHAMFER   * l_ch
          + W_GAP       * ramp * l_gap
          + W_AXIAL     * ramp * l_ax
          + W_SW        * ramp * l_sw
          + W_REPULSION * l_rep
          + W_AUX_OCC   * l_aux
          + W_OFFSET    * l_off
          + w_proj_eff  * l_proj)

    bd = {
        'total':     float(loss.detach().cpu()),
        'chamfer':   float(l_ch.detach().cpu()),
        'gap':       float(l_gap.detach().cpu()),
        'axial':     float(l_ax.detach().cpu()),
        'sw':        float(l_sw.detach().cpu()),
        'repulsion': float(l_rep.detach().cpu()),
        'aux_occ':   float(l_aux.detach().cpu()),
        'offset':    float(l_off.detach().cpu()),
        'proj':      float(l_proj.detach().cpu()),
        'w_proj_eff': float(w_proj_eff),
        'ramp':      float(ramp),
    }
    return loss, bd

print('Loss functions ready (gap-preserving suite + projection-consistency).')


Loss functions ready (gap-preserving suite + projection-consistency).


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — TRAINING UTILITIES (checkpoint I/O, collate, eval, metrics)
# ══════════════════════════════════════════════════════════════════════════════

def save_checkpoint(path, model, optimizer, scheduler, scaler,
                    epoch, best_val, history, tag='v10_generator'):
    state = {
        'model':           model.state_dict(),
        'optimizer':       optimizer.state_dict(),
        'scheduler':       scheduler.state_dict() if scheduler else None,
        'scaler':          scaler.state_dict() if scaler else None,
        'epoch':           epoch,
        'best_val_loss':   best_val,
        'training_history': history,
        'config':          {'version': tag},
        'timestamp':       datetime.utcnow().isoformat(),
    }
    tmp = path.with_suffix('.pth.tmp')
    torch.save(state, tmp)
    tmp.replace(path)


def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None):
    state = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(state['model'])
    if optimizer and state.get('optimizer'): optimizer.load_state_dict(state['optimizer'])
    if scheduler and state.get('scheduler'): scheduler.load_state_dict(state['scheduler'])
    if scaler    and state.get('scaler'):    scaler.load_state_dict(state['scaler'])
    start = state['epoch'] + 1
    best  = state.get('best_val_loss', float('inf'))
    hist  = state.get('training_history', [])
    print(f'  Resumed from epoch {start} | best_val_loss: {best:.4f}')
    return start, best, hist


def collate_keep_strings(batch):
    """Default collate that keeps patient_id strings as a Python list."""
    out = {}
    for k in batch[0]:
        vals = [b[k] for b in batch]
        out[k] = torch.stack(vals, 0) if isinstance(vals[0], torch.Tensor) else vals
    return out


def move_to_device(batch):
    return {k: (v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v)
            for k, v in batch.items()}


def chamfer_metrics_np(pred, gt):
    pred = np.asarray(pred, np.float32)
    gt   = np.asarray(gt, np.float32)
    d = np.linalg.norm(pred[:, None] - gt[None], axis=-1)
    d_pg = d.min(axis=1)
    d_gp = d.min(axis=0)
    def fs(th):
        p = (d_pg <= th).mean(); r = (d_gp <= th).mean()
        return 2*p*r/(p+r) if (p+r) > 0 else 0.0
    return {
        'chamfer_mm':   float(0.5*(d_pg.mean()+d_gp.mean())),
        'fscore_1mm':   float(fs(1.0)),
        'fscore_2mm':   float(fs(2.0)),
        'fscore_5mm':   float(fs(5.0)),
        'hausdorff_95': float(max(np.percentile(d_pg,95), np.percentile(d_gp,95))),
    }


@torch.no_grad()
def evaluate_generator(model, loader, max_samples=MAX_EVAL_SAMPLES, current_epoch=0):
    """Validation pass for Phase 1 (generator only)."""
    model.eval()
    acc = {k: 0.0 for k in ('total','chamfer','gap','axial','sw','repulsion','aux_occ','offset','proj')}
    n_bat = 0; metrics = []; n_done = 0
    for batch in loader:
        batch  = move_to_device(batch)
        output = model(batch['drr_ap'], batch['drr_lp'],
                       batch['P_ap'], batch['P_lp'],
                       batch['center'], batch['scale'])
        _, bd = total_loss(output, batch, current_epoch)
        for k in acc:
            acc[k] += bd.get(k, 0.0)
        n_bat += 1
        if n_done < max_samples:
            pred = output['pred_world'].cpu().numpy()
            gt   = batch['gt_ppc_world'].cpu().numpy()
            for b in range(pred.shape[0]):
                if n_done >= max_samples: break
                metrics.append(chamfer_metrics_np(pred[b], gt[b]))
                n_done += 1
    result = {k: acc[k]/max(1, n_bat) for k in acc}
    if metrics:
        result.update({
            'mean_mm':      float(np.mean([m['chamfer_mm'] for m in metrics])),
            'std_mm':       float(np.std([m['chamfer_mm'] for m in metrics])),
            'fscore_1mm':   float(np.mean([m['fscore_1mm'] for m in metrics])),
            'fscore_2mm':   float(np.mean([m['fscore_2mm'] for m in metrics])),
            'fscore_5mm':   float(np.mean([m['fscore_5mm'] for m in metrics])),
            'hausdorff_95': float(np.mean([m['hausdorff_95'] for m in metrics])),
            'n_patients':   n_done,
        })
    else:
        result.update(dict(mean_mm=float('inf'), std_mm=0.0,
                           fscore_1mm=0.0, fscore_2mm=0.0,
                           fscore_5mm=0.0, hausdorff_95=float('inf'), n_patients=0))
    return result

print('Training utilities ready.')


Training utilities ready.


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — PHASE 1: TRAIN PPCNetV10 (the v8 generator) FROM SCRATCH
#   This is the proven path to ~2.21 mm Chamfer. Resumes from
#   gen_latest.pth if present. Saves:
#     gen_latest.pth       — every epoch (resume)
#     gen_best.pth         — best val total loss
#     gen_best_chamfer.pth — best mean Chamfer (USE THIS for downstream)
# ══════════════════════════════════════════════════════════════════════════════

train_ds = LumbarDatasetV10(train_ids, augment=True)
val_ds   = LumbarDatasetV10(val_ids,  augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_keep_strings, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_keep_strings, persistent_workers=True)

print(f'Train: {len(train_ds)} patients → {len(train_loader)} batches/epoch')
print(f'Val  : {len(val_ds)}   patients → {len(val_loader)}   batches/epoch')

generator = PPCNetV10().to(device)
optimizer = torch.optim.AdamW(generator.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP and device.type == 'cuda')

warmup_steps = max(1, WARMUP_EPOCHS * len(train_loader))
total_steps  = max(1, EPOCHS * len(train_loader))

def lr_lambda(step):
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

start_epoch   = 0
best_val_loss = float('inf')
best_chamfer  = float('inf')
history       = []

if GEN_CKPT_PATH.exists():
    print('Resuming generator from checkpoint …')
    start_epoch, best_val_loss, history = load_checkpoint(
        GEN_CKPT_PATH, generator, optimizer, scheduler, scaler)
    best_chamfer = min(
        (r['val'].get('mean_mm', float('inf')) for r in history),
        default=float('inf'))
else:
    print('No checkpoint — starting fresh.')

print(f'Generator params : {count_params(generator)/1e6:.1f} M')
print(f'Start epoch      : {start_epoch + 1}')

for epoch in range(start_epoch, EPOCHS):
    generator.train()
    acc = {k: 0.0 for k in ('total','chamfer','gap','axial','sw',
                            'repulsion','aux_occ','offset','proj')}

    for bi, batch in enumerate(train_loader, start=1):
        batch = move_to_device(batch)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP and device.type == 'cuda'):
            output = generator(batch['drr_ap'], batch['drr_lp'],
                               batch['P_ap'], batch['P_lp'],
                               batch['center'], batch['scale'])
            loss, bd = total_loss(output, batch, current_epoch=epoch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(generator.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        for k in acc: acc[k] += bd.get(k, 0.0)

        if bi % 100 == 0 or bi == len(train_loader):
            lr_now = optimizer.param_groups[0]['lr']
            print(f"  [Ep {epoch+1:3d}/{EPOCHS}] {bi:3d}/{len(train_loader)} "
                  f"loss={bd['total']:.4f} ch={bd['chamfer']:.4f} "
                  f"gap={bd['gap']:.4f} ax={bd['axial']:.4f} "
                  f"proj={bd['proj']:.4f} ramp={bd['ramp']:.2f} lr={lr_now:.2e}")

    train_stats = {k: acc[k]/max(1, len(train_loader)) for k in acc}
    val_stats   = evaluate_generator(generator, val_loader, current_epoch=epoch)

    rec = {'epoch': epoch+1, 'train': train_stats, 'val': val_stats,
           'lr': optimizer.param_groups[0]['lr']}
    history.append(rec)
    append_training_log(GEN_LOG, rec)

    save_checkpoint(GEN_CKPT_PATH, generator, optimizer, scheduler, scaler,
                    epoch, best_val_loss, history, tag='v10_generator')

    print(f"[Epoch {epoch+1:3d}/{EPOCHS}] "
          f"train={train_stats['total']:.4f} val={val_stats['total']:.4f}")
    print(f"  Chamfer={val_stats['mean_mm']:.3f}±{val_stats['std_mm']:.3f} mm "
          f"F@1={val_stats['fscore_1mm']:.3f} F@2={val_stats['fscore_2mm']:.3f} "
          f"F@5={val_stats['fscore_5mm']:.3f} HD95={val_stats['hausdorff_95']:.3f}")

    if val_stats['total'] < best_val_loss:
        best_val_loss = val_stats['total']
        save_checkpoint(GEN_BEST_CKPT, generator, optimizer, scheduler, scaler,
                        epoch, best_val_loss, history, tag='v10_generator')
        print(f"  ✓ New best val_loss: {best_val_loss:.4f}")

    if val_stats['mean_mm'] < best_chamfer:
        best_chamfer = val_stats['mean_mm']
        save_checkpoint(GEN_BEST_CH_CKPT, generator, optimizer, scheduler, scaler,
                        epoch, best_val_loss, history, tag='v10_generator')
        print(f"  ✓ New best Chamfer: {best_chamfer:.3f} mm")

print(f'\nPhase 1 (generator) complete. Best Chamfer: {best_chamfer:.3f} mm')


Train: 829 patients → 415 batches/epoch
Val  : 103   patients → 52   batches/epoch
No checkpoint — starting fresh.
Generator params : 21.8 M
Start epoch      : 1


/apps/app/see_all_ai/seeallai/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


  [Ep   1/250] 100/415 loss=0.0540 ch=0.0000 gap=0.9923 ax=0.7183 proj=0.0000 ramp=0.00 lr=1.95e-06
  [Ep   1/250] 200/415 loss=0.0271 ch=0.0000 gap=0.9902 ax=0.7205 proj=0.0000 ramp=0.00 lr=3.87e-06
  [Ep   1/250] 300/415 loss=0.0204 ch=0.0000 gap=0.9900 ax=0.6787 proj=0.0000 ramp=0.00 lr=5.80e-06
  [Ep   1/250] 400/415 loss=0.0207 ch=0.0000 gap=0.9896 ax=0.8239 proj=0.0000 ramp=0.00 lr=7.73e-06
  [Ep   1/250] 415/415 loss=0.0149 ch=0.0000 gap=0.9897 ax=0.6425 proj=0.0000 ramp=0.00 lr=8.02e-06
[Epoch   1/250] train=2.2664 val=410.9156
  Chamfer=11.617±0.658 mm F@1=0.013 F@2=0.083 F@5=0.303 HD95=43.252
  ✓ New best val_loss: 410.9156
  ✓ New best Chamfer: 11.617 mm
  [Ep   2/250] 100/415 loss=0.0171 ch=0.0000 gap=0.9885 ax=0.6663 proj=0.0000 ramp=0.00 lr=9.95e-06
  [Ep   2/250] 200/415 loss=0.0336 ch=0.0000 gap=0.9890 ax=0.9085 proj=0.0000 ramp=0.00 lr=1.19e-05
  [Ep   2/250] 300/415 loss=0.0145 ch=0.0000 gap=0.9855 ax=0.4774 proj=0.0000 ramp=0.00 lr=1.38e-05
  [Ep   2/250] 400/415 los

  [Ep  13/250] 400/415 loss=0.0147 ch=0.0000 gap=0.9554 ax=0.2245 proj=0.0000 ramp=0.00 lr=8.00e-05
  [Ep  13/250] 415/415 loss=0.0137 ch=0.0000 gap=0.9541 ax=0.3155 proj=0.0000 ramp=0.00 lr=8.00e-05
[Epoch  13/250] train=0.3135 val=106.1432
  Chamfer=4.770±0.347 mm F@1=0.053 F@2=0.312 F@5=0.791 HD95=34.695
  [Ep  14/250] 100/415 loss=0.0143 ch=0.0000 gap=0.9419 ax=0.3062 proj=0.0000 ramp=0.00 lr=8.00e-05
  [Ep  14/250] 200/415 loss=0.0127 ch=0.0000 gap=0.9413 ax=0.3308 proj=0.0000 ramp=0.00 lr=8.00e-05
  [Ep  14/250] 300/415 loss=0.0136 ch=0.0000 gap=0.9544 ax=0.2683 proj=0.0000 ramp=0.00 lr=8.00e-05
  [Ep  14/250] 400/415 loss=0.0130 ch=0.0000 gap=0.9400 ax=0.2528 proj=0.0000 ramp=0.00 lr=7.99e-05
  [Ep  14/250] 415/415 loss=0.0163 ch=0.0000 gap=0.9512 ax=0.2230 proj=0.0000 ramp=0.00 lr=7.99e-05
[Epoch  14/250] train=0.3571 val=30.5211
  Chamfer=2.908±0.344 mm F@1=0.063 F@2=0.363 F@5=0.882 HD95=8.137
  [Ep  15/250] 100/415 loss=0.0168 ch=0.0000 gap=0.9438 ax=0.1988 proj=0.0000 ramp=0

  [Ep  26/250] 400/415 loss=0.2192 ch=0.0000 gap=0.9188 ax=0.1356 proj=0.0000 ramp=0.40 lr=7.91e-05
  [Ep  26/250] 415/415 loss=0.2028 ch=0.0000 gap=0.9192 ax=0.0539 proj=0.0000 ramp=0.40 lr=7.91e-05
[Epoch  26/250] train=0.3957 val=19.5854
  Chamfer=2.341±0.420 mm F@1=0.094 F@2=0.491 F@5=0.951 HD95=5.157
  ✓ New best val_loss: 19.5854
  [Ep  27/250] 100/415 loss=2.8175 ch=2.5861 gap=0.9261 ax=0.0842 proj=0.0000 ramp=0.44 lr=7.91e-05
  [Ep  27/250] 200/415 loss=0.2320 ch=0.0000 gap=0.9245 ax=0.0784 proj=0.0000 ramp=0.44 lr=7.91e-05
  [Ep  27/250] 300/415 loss=0.2353 ch=0.0000 gap=0.9345 ax=0.0895 proj=0.0000 ramp=0.44 lr=7.90e-05
  [Ep  27/250] 400/415 loss=0.2290 ch=0.0000 gap=0.9223 ax=0.0766 proj=0.0000 ramp=0.44 lr=7.90e-05
  [Ep  27/250] 415/415 loss=0.2325 ch=0.0000 gap=0.9172 ax=0.1026 proj=0.0000 ramp=0.44 lr=7.90e-05
[Epoch  27/250] train=0.4093 val=19.9058
  Chamfer=2.369±0.427 mm F@1=0.092 F@2=0.483 F@5=0.949 HD95=5.257
  [Ep  28/250] 100/415 loss=0.2507 ch=0.0000 gap=0.9167

...

[Epoch 188/250] train=0.4814 val=31.7241
  Chamfer=2.510±0.443 mm F@1=0.117 F@2=0.483 F@5=0.911 HD95=8.650
  [Ep 189/250] 100/415 loss=2.8533 ch=2.3719 gap=0.8324 ax=0.1232 proj=0.0000 ramp=1.00 lr=1.24e-05
  [Ep 189/250] 200/415 loss=3.4529 ch=2.9998 gap=0.8163 ax=0.0781 proj=0.0000 ramp=1.00 lr=1.23e-05
  [Ep 189/250] 300/415 loss=0.4343 ch=0.0000 gap=0.8287 ax=0.0160 proj=0.0000 ramp=1.00 lr=1.22e-05
  [Ep 189/250] 400/415 loss=0.4093 ch=0.0000 gap=0.7800 ax=0.0135 proj=0.0000 ramp=1.00 lr=1.21e-05
  [Ep 189/250] 415/415 loss=0.4049 ch=0.0000 gap=0.7728 ax=0.0147 proj=0.0000 ramp=1.00 lr=1.21e-05
[Epoch 189/250] train=0.4806 val=31.6000
  Chamfer=2.508±0.444 mm F@1=0.116 F@2=0.483 F@5=0.912 HD95=8.615
  [Ep 190/250] 100/415 loss=0.3846 ch=0.0000 gap=0.7280 ax=0.0151 proj=0.0000 ramp=1.00 lr=1.20e-05
  [Ep 190/250] 200/415 loss=0.4102 ch=0.0000 gap=0.7796 ax=0.0169 proj=0.0000 ramp=1.00 lr=1.19e-05
  [Ep 190/250] 300/415 loss=0.4063 ch=0.0000 gap=0.7709 ax=0.0139 proj=0.0000 ramp=1.0

  [Ep 202/250] 300/415 loss=0.3919 ch=0.0000 gap=0.7481 ax=0.0124 proj=0.0000 ramp=1.00 lr=7.72e-06
  [Ep 202/250] 400/415 loss=0.4407 ch=0.0000 gap=0.8395 ax=0.0152 proj=0.0000 ramp=1.00 lr=7.65e-06
  [Ep 202/250] 415/415 loss=0.4088 ch=0.0000 gap=0.7791 ax=0.0122 proj=0.0000 ramp=1.00 lr=7.64e-06
[Epoch 202/250] train=0.4754 val=31.1610
  Chamfer=2.500±0.443 mm F@1=0.116 F@2=0.483 F@5=0.913 HD95=8.476
  [Ep 203/250] 100/415 loss=0.4029 ch=0.0000 gap=0.7659 ax=0.0152 proj=0.0000 ramp=1.00 lr=7.57e-06
  [Ep 203/250] 200/415 loss=0.4064 ch=0.0000 gap=0.7757 ax=0.0122 proj=0.0000 ramp=1.00 lr=7.49e-06
  [Ep 203/250] 300/415 loss=0.4038 ch=0.0000 gap=0.7704 ax=0.0128 proj=0.0000 ramp=1.00 lr=7.42e-06
  [Ep 203/250] 400/415 loss=0.4022 ch=0.0000 gap=0.7650 ax=0.0133 proj=0.0000 ramp=1.00 lr=7.35e-06
  [Ep 203/250] 415/415 loss=0.4075 ch=0.0000 gap=0.7805 ax=0.0098 proj=0.0000 ramp=1.00 lr=7.33e-06
[Epoch 203/250] train=0.4749 val=31.8267
  Chamfer=2.511±0.445 mm F@1=0.116 F@2=0.483 F@5=0.9

  [Ep 216/250] 100/415 loss=0.4264 ch=0.0000 gap=0.8138 ax=0.0107 proj=0.0000 ramp=1.00 lr=4.07e-06
  [Ep 216/250] 200/415 loss=0.4010 ch=0.0000 gap=0.7634 ax=0.0108 proj=0.0000 ramp=1.00 lr=4.01e-06
  [Ep 216/250] 300/415 loss=0.4205 ch=0.0000 gap=0.8020 ax=0.0130 proj=0.0000 ramp=1.00 lr=3.96e-06
  [Ep 216/250] 400/415 loss=0.3891 ch=0.0000 gap=0.7407 ax=0.0118 proj=0.0000 ramp=1.00 lr=3.90e-06
  [Ep 216/250] 415/415 loss=0.4155 ch=0.0000 gap=0.7889 ax=0.0154 proj=0.0000 ramp=1.00 lr=3.90e-06
[Epoch 216/250] train=0.4716 val=31.4292
  Chamfer=2.507±0.443 mm F@1=0.116 F@2=0.481 F@5=0.912 HD95=8.532
  [Ep 217/250] 100/415 loss=3.4363 ch=2.9841 gap=0.8120 ax=0.0793 proj=0.0000 ramp=1.00 lr=3.84e-06
  [Ep 217/250] 200/415 loss=0.4155 ch=0.0000 gap=0.7943 ax=0.0107 proj=0.0000 ramp=1.00 lr=3.79e-06
  [Ep 217/250] 300/415 loss=0.4049 ch=0.0000 gap=0.7723 ax=0.0095 proj=0.0000 ramp=1.00 lr=3.74e-06
  [Ep 217/250] 400/415 loss=0.3744 ch=0.0000 gap=0.7113 ax=0.0110 proj=0.0000 ramp=1.00 lr=3.

  [Ep 229/250] 415/415 loss=0.3886 ch=0.0000 gap=0.7407 ax=0.0095 proj=0.0000 ramp=1.00 lr=1.50e-06
[Epoch 229/250] train=0.4686 val=31.2549
  Chamfer=2.505±0.443 mm F@1=0.116 F@2=0.481 F@5=0.912 HD95=8.481
  [Ep 230/250] 100/415 loss=0.3802 ch=0.0000 gap=0.7232 ax=0.0117 proj=0.0000 ramp=1.00 lr=1.47e-06
  [Ep 230/250] 200/415 loss=0.4087 ch=0.0000 gap=0.7824 ax=0.0097 proj=0.0000 ramp=1.00 lr=1.43e-06
  [Ep 230/250] 300/415 loss=0.4128 ch=0.0000 gap=0.7870 ax=0.0104 proj=0.0000 ramp=1.00 lr=1.40e-06
  [Ep 230/250] 400/415 loss=0.4179 ch=0.0000 gap=0.7975 ax=0.0120 proj=0.0000 ramp=1.00 lr=1.37e-06
  [Ep 230/250] 415/415 loss=0.3942 ch=0.0000 gap=0.7485 ax=0.0117 proj=0.0000 ramp=1.00 lr=1.36e-06
[Epoch 230/250] train=0.4692 val=31.3647
  Chamfer=2.507±0.442 mm F@1=0.116 F@2=0.481 F@5=0.911 HD95=8.517
  [Ep 231/250] 100/415 loss=0.3752 ch=0.0000 gap=0.7163 ax=0.0087 proj=0.0000 ramp=1.00 lr=1.33e-06
  [Ep 231/250] 200/415 loss=0.4669 ch=0.0000 gap=0.8753 ax=0.0121 proj=0.0000 ramp=1.0

  [Ep 243/250] 200/415 loss=0.3822 ch=0.0000 gap=0.7283 ax=0.0099 proj=0.0000 ramp=1.00 lr=1.94e-07
  [Ep 243/250] 300/415 loss=0.3916 ch=0.0000 gap=0.7474 ax=0.0104 proj=0.0000 ramp=1.00 lr=1.81e-07
  [Ep 243/250] 400/415 loss=0.3931 ch=0.0000 gap=0.7511 ax=0.0088 proj=0.0000 ramp=1.00 lr=1.70e-07
  [Ep 243/250] 415/415 loss=0.4160 ch=0.0000 gap=0.7889 ax=0.0129 proj=0.0000 ramp=1.00 lr=1.68e-07
[Epoch 243/250] train=0.4679 val=31.3458
  Chamfer=2.505±0.443 mm F@1=0.116 F@2=0.481 F@5=0.912 HD95=8.505
  [Ep 244/250] 100/415 loss=0.4026 ch=0.0000 gap=0.7697 ax=0.0109 proj=0.0000 ramp=1.00 lr=1.56e-07
  [Ep 244/250] 200/415 loss=0.3979 ch=0.0000 gap=0.7593 ax=0.0095 proj=0.0000 ramp=1.00 lr=1.46e-07
  [Ep 244/250] 300/415 loss=0.3798 ch=0.0000 gap=0.7225 ax=0.0085 proj=0.0000 ramp=1.00 lr=1.35e-07
  [Ep 244/250] 400/415 loss=0.4203 ch=0.0000 gap=0.8038 ax=0.0129 proj=0.0000 ramp=1.00 lr=1.25e-07
  [Ep 244/250] 415/415 loss=0.4266 ch=0.0000 gap=0.8101 ax=0.0123 proj=0.0000 ramp=1.00 lr=1.

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 — PHASE 2 (OPTIONAL): SN-GAN REFINEMENT ON TOP OF FROZEN GENERATOR
#   Only run AFTER Phase 1 has saved gen_best_chamfer.pth (or after manually
#   placing a v10 generator checkpoint there). This is the v9 refiner exactly.
# ══════════════════════════════════════════════════════════════════════════════

# ── DGCNN-style edge convolutions for refiner + spectral-norm discriminator ──

def knn_graph(x, k=10):
    d = torch.cdist(x, x)
    _, idx = d.topk(k+1, dim=-1, largest=False)
    return idx[:, :, 1:]


def get_edge_features(x, idx):
    B, N, C = x.shape
    k = idx.shape[-1]
    nb = torch.gather(x.unsqueeze(2).expand(B, N, N, C), 2,
                      idx.unsqueeze(-1).expand(B, N, k, C))
    return torch.cat([x.unsqueeze(2).expand_as(nb), nb - x.unsqueeze(2).expand_as(nb)], -1)


class EdgeConv(nn.Module):
    """Plain EdgeConv for the refiner (generator side)."""
    def __init__(self, in_c, out_c, k=10):
        super().__init__()
        self.k = k
        self.mlp = nn.Sequential(
            nn.Linear(in_c * 2, out_c), nn.LeakyReLU(0.2),
            nn.Linear(out_c, out_c),    nn.LeakyReLU(0.2))
    def forward(self, x, idx=None):
        if idx is None: idx = knn_graph(x, self.k)
        return self.mlp(get_edge_features(x, idx)).max(dim=2)[0]


class EdgeConvSN(nn.Module):
    """EdgeConv with spectral normalization for the discriminator."""
    def __init__(self, in_c, out_c, k=10):
        super().__init__()
        self.k = k
        self.mlp = nn.Sequential(
            nn.utils.spectral_norm(nn.Linear(in_c * 2, out_c)), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Linear(out_c, out_c)),    nn.LeakyReLU(0.2))
    def forward(self, x, idx=None):
        if idx is None: idx = knn_graph(x, self.k)
        return self.mlp(get_edge_features(x, idx)).max(dim=2)[0]


class PointDiscriminator(nn.Module):
    """SN-GAN discriminator. Input: (B, D_SUBSAMPLE, 3) in LOCAL [-1,1]."""
    def __init__(self, k=10):
        super().__init__()
        self.ec1 = EdgeConvSN(3,   64, k)
        self.ec2 = EdgeConvSN(64, 128, k)
        self.global_mlp = nn.Sequential(
            nn.utils.spectral_norm(nn.Linear(64+128, 256)), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Linear(256,    128)), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Linear(128,      1)))
    def forward(self, x):
        f1 = self.ec1(x)
        f2 = self.ec2(f1)
        g  = torch.cat([f1, f2], -1).max(dim=1)[0]
        return self.global_mlp(g)


def subsample_points(pts, n=D_SUBSAMPLE):
    B, N, C = pts.shape
    idx = torch.randint(0, N, (B, n), device=pts.device)
    return torch.gather(pts, 1, idx.unsqueeze(-1).expand(B, n, C))


class PointRefiner(nn.Module):
    """Takes coarse points (B,N,3) → refined points + per-point confidence."""
    def __init__(self, k=16):
        super().__init__()
        self.ec1 = EdgeConv(3,   64, k)
        self.ec2 = EdgeConv(64, 128, k)
        self.offset_head = nn.Sequential(
            nn.Linear(64+128+3, 128), nn.GELU(),
            nn.Linear(128, 64),       nn.GELU(),
            nn.Linear(64, 3))
        self.conf_head = nn.Sequential(
            nn.Linear(64+128+3, 64), nn.GELU(),
            nn.Linear(64, 1),        nn.Sigmoid())

    def forward(self, pts_local):
        f1 = self.ec1(pts_local)
        f2 = self.ec2(f1)
        feat = torch.cat([f1, f2, pts_local], -1)
        offset = 0.08 * torch.tanh(self.offset_head(feat))
        conf   = self.conf_head(feat).squeeze(-1)
        refined = torch.clamp(pts_local + offset, -1.0, 1.0)
        return refined, conf, offset


def chamfer_loss_plain(pred, gt):
    d2 = pairwise_sqdist(pred, gt)
    return 0.5 * (d2.min(2)[0].mean() + d2.min(1)[0].mean())


def confidence_loss(conf, pred_local, gap_occ_gt):
    B, N, _ = pred_local.shape
    occ = gap_occ_gt.unsqueeze(1)
    grid = torch.stack([pred_local[..., 2], pred_local[..., 1],
                        pred_local[..., 0]], dim=-1).view(B, N, 1, 1, 3)
    occ_at = F.grid_sample(occ, grid, mode='bilinear', align_corners=True,
                           padding_mode='border').view(B, N)
    return F.binary_cross_entropy(conf, occ_at.detach())


# ── Load frozen v10 generator from Phase 1 ──────────────────────────────────
gen_ckpt_to_use = GEN_BEST_CH_CKPT if GEN_BEST_CH_CKPT.exists() else GEN_BEST_CKPT
if not gen_ckpt_to_use.exists():
    gen_ckpt_to_use = GEN_CKPT_PATH

if not gen_ckpt_to_use.exists():
    raise FileNotFoundError(
        f'No generator checkpoint found in {CKPT_DIR}. Run Cell 8 first.')

print(f'Loading frozen generator from: {gen_ckpt_to_use.name}')
frozen_gen = PPCNetV10().to(device)
gen_state = torch.load(gen_ckpt_to_use, map_location=device, weights_only=False)
frozen_gen.load_state_dict(gen_state['model'])
frozen_gen.eval()
for p in frozen_gen.parameters(): p.requires_grad_(False)
print(f'  Loaded generator (epoch {gen_state["epoch"]+1})')

# ── Refiner + Discriminator ─────────────────────────────────────────────────
refiner       = PointRefiner(k=16).to(device)
discriminator = PointDiscriminator(k=10).to(device)
print(f'Refiner params       : {count_params(refiner)/1e6:.2f} M')
print(f'Discriminator params : {count_params(discriminator)/1e6:.2f} M')

opt_G = torch.optim.AdamW(refiner.parameters(),       lr=LR_G, betas=(0.5, 0.999), weight_decay=1e-5)
opt_D = torch.optim.AdamW(discriminator.parameters(), lr=LR_D, betas=(0.5, 0.999), weight_decay=1e-5)

# Reuse the same train/val loaders from Cell 8
print(f'\n{"="*60}\nStarting GAN refinement training...\n{"="*60}')

best_chamfer_gan = float('inf')
history_gan = []

def save_gan_ckpt(path, epoch, best_ch, hist):
    state = {
        'refiner':       refiner.state_dict(),
        'discriminator': discriminator.state_dict(),
        'opt_G':         opt_G.state_dict(),
        'opt_D':         opt_D.state_dict(),
        'epoch':         epoch,
        'best_chamfer':  best_ch,
        'history':       hist,
        'config':        {'version': 'v10_gan_refiner'},
        'timestamp':     datetime.utcnow().isoformat(),
    }
    tmp = path.with_suffix('.pth.tmp')
    torch.save(state, tmp)
    tmp.replace(path)


# Resume GAN if possible
gan_start_epoch = 0
if GAN_CKPT_PATH.exists():
    g_state = torch.load(GAN_CKPT_PATH, map_location=device, weights_only=False)
    refiner.load_state_dict(g_state['refiner'])
    discriminator.load_state_dict(g_state['discriminator'])
    opt_G.load_state_dict(g_state['opt_G'])
    opt_D.load_state_dict(g_state['opt_D'])
    gan_start_epoch = g_state['epoch'] + 1
    best_chamfer_gan = g_state.get('best_chamfer', float('inf'))
    history_gan = g_state.get('history', [])
    print(f'Resumed GAN from epoch {gan_start_epoch} (best_chamfer={best_chamfer_gan:.3f} mm)')

for epoch in range(gan_start_epoch, GAN_EPOCHS):
    refiner.train(); discriminator.train()
    acc = {'d_loss': 0.0, 'g_loss': 0.0, 'chamfer': 0.0, 'gap': 0.0,
           'axial': 0.0, 'conf': 0.0}
    n_bat = 0

    for bi, batch in enumerate(train_loader, start=1):
        batch = move_to_device(batch)

        # Coarse predictions from frozen generator
        with torch.no_grad():
            v_out = frozen_gen(batch['drr_ap'], batch['drr_lp'],
                               batch['P_ap'], batch['P_lp'],
                               batch['center'], batch['scale'])
            coarse_local = v_out['pred_local'].detach()

        gt_local = batch['gt_ppc_local']

        # ── Discriminator step (N_CRITIC out of every N_CRITIC+1 batches) ──
        if bi % (N_CRITIC + 1) != 0:
            opt_D.zero_grad(set_to_none=True)
            with torch.no_grad():
                refined, _, _ = refiner(coarse_local)
            gt_sub   = subsample_points(gt_local,         D_SUBSAMPLE)
            fake_sub = subsample_points(refined.detach(), D_SUBSAMPLE)
            d_real = discriminator(gt_sub)
            d_fake = discriminator(fake_sub)
            d_loss = F.relu(1.0 - d_real).mean() + F.relu(1.0 + d_fake).mean()
            d_loss.backward()
            torch.nn.utils.clip_grad_norm_(discriminator.parameters(), GRAD_CLIP)
            opt_D.step()
            acc['d_loss'] += float(d_loss.detach().cpu())

        # ── Refiner (generator) step ──────────────────────────────────────
        else:
            opt_G.zero_grad(set_to_none=True)
            refined, conf, offset = refiner(coarse_local)
            ref_sub = subsample_points(refined, D_SUBSAMPLE)
            g_adv  = -discriminator(ref_sub).mean()
            g_ch   = chamfer_loss_plain(refined, gt_local)
            g_gap  = gap_penalty_loss(refined, batch['gap_occ'])
            g_ax   = soft_axial_density_loss(refined, gt_local)
            g_sw   = sliced_wasserstein_loss(refined, gt_local)
            g_conf = confidence_loss(conf, coarse_local, batch['gap_occ'])
            g_off  = offset.abs().mean()
            g_loss = (W_ADV       * g_adv
                    + W_CHAMFER_R * g_ch
                    + W_GAP_R     * g_gap
                    + W_AXIAL_R   * g_ax
                    + W_SW_R      * g_sw
                    + 0.2         * g_conf
                    + 0.001       * g_off)
            g_loss.backward()
            torch.nn.utils.clip_grad_norm_(refiner.parameters(), GRAD_CLIP)
            opt_G.step()
            acc['g_loss']  += float(g_loss.detach().cpu())
            acc['chamfer'] += float(g_ch.detach().cpu())
            acc['gap']     += float(g_gap.detach().cpu())
            acc['axial']   += float(g_ax.detach().cpu())
            acc['conf']    += float(g_conf.detach().cpu())

        n_bat += 1
        if bi % 100 == 0 or bi == len(train_loader):
            print(f"  [GAN Ep {epoch+1:3d}/{GAN_EPOCHS}] {bi:3d}/{len(train_loader)} "
                  f"D={acc['d_loss']/max(1,n_bat):.4f} "
                  f"G={acc['g_loss']/max(1,n_bat):.4f} "
                  f"gap={acc['gap']/max(1,n_bat):.4f}")

    # ── Validation with confidence filtering ───────────────────────────────
    refiner.eval()
    metrics = []; n_done = 0
    with torch.no_grad():
        for batch in val_loader:
            if n_done >= MAX_EVAL_SAMPLES: break
            batch = move_to_device(batch)
            v_out = frozen_gen(batch['drr_ap'], batch['drr_lp'],
                               batch['P_ap'], batch['P_lp'],
                               batch['center'], batch['scale'])
            refined, conf, _ = refiner(v_out['pred_local'])
            ref_world = local_to_world(refined, batch['center'], batch['scale'])
            pred = ref_world.cpu().numpy()
            gt_w = batch['gt_ppc_world'].cpu().numpy()

            for b in range(pred.shape[0]):
                if n_done >= MAX_EVAL_SAMPLES: break
                c = conf[b].cpu().numpy()
                pts = pred[b]
                mask = c > 0.3
                if mask.sum() > 100:
                    good_pts = pts[mask]
                    n_need = N_POINTS - len(good_pts)
                    if n_need > 0:
                        extra = good_pts[np.random.choice(len(good_pts), n_need, replace=True)]
                        pts = np.concatenate([good_pts, extra], 0)
                    else:
                        pts = good_pts[:N_POINTS]
                metrics.append(chamfer_metrics_np(pts, gt_w[b]))
                n_done += 1

    if metrics:
        mean_ch = float(np.mean([m['chamfer_mm']   for m in metrics]))
        f1      = float(np.mean([m['fscore_1mm']   for m in metrics]))
        f2      = float(np.mean([m['fscore_2mm']   for m in metrics]))
        f5      = float(np.mean([m['fscore_5mm']   for m in metrics]))
        hd      = float(np.mean([m['hausdorff_95'] for m in metrics]))
        print(f"[GAN Epoch {epoch+1}] Chamfer={mean_ch:.3f} mm  "
              f"F@1={f1:.3f}  F@2={f2:.3f}  F@5={f5:.3f}  HD95={hd:.3f}")

        rec = {'epoch': epoch+1, 'chamfer_mm': mean_ch,
               'f1': f1, 'f2': f2, 'f5': f5, 'hd95': hd}
        history_gan.append(rec)
        append_training_log(GAN_LOG, rec)

        save_gan_ckpt(GAN_CKPT_PATH, epoch, best_chamfer_gan, history_gan)

        if mean_ch < best_chamfer_gan:
            best_chamfer_gan = mean_ch
            save_gan_ckpt(GAN_BEST_CKPT, epoch, best_chamfer_gan, history_gan)
            print(f"  ✓ New best GAN Chamfer: {best_chamfer_gan:.3f} mm")

print(f'\nPhase 2 (GAN refinement) complete. Best Chamfer: {best_chamfer_gan:.3f} mm')


NameError: name 'nn' is not defined

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ── Helper functions ──────────────────────────────────────────────────────────

def knn_graph(x, k=10):
    d = torch.cdist(x, x)
    _, idx = d.topk(k+1, dim=-1, largest=False)
    return idx[:, :, 1:]


def get_edge_features(x, idx):
    B, N, C = x.shape
    k = idx.shape[-1]
    nb = torch.gather(x.unsqueeze(2).expand(B, N, N, C), 2,
                      idx.unsqueeze(-1).expand(B, N, k, C))
    return torch.cat([x.unsqueeze(2).expand_as(nb), nb - x.unsqueeze(2).expand_as(nb)], -1)


# ── Model classes ─────────────────────────────────────────────────────────────

class EdgeConv(nn.Module):
    def __init__(self, in_c, out_c, k=10):
        super().__init__()
        self.k = k
        self.mlp = nn.Sequential(
            nn.Linear(in_c * 2, out_c), nn.LeakyReLU(0.2),
            nn.Linear(out_c, out_c),    nn.LeakyReLU(0.2))
    def forward(self, x, idx=None):
        if idx is None: idx = knn_graph(x, self.k)
        return self.mlp(get_edge_features(x, idx)).max(dim=2)[0]


class EdgeConvSN(nn.Module):
    def __init__(self, in_c, out_c, k=10):
        super().__init__()
        self.k = k
        self.mlp = nn.Sequential(
            nn.utils.spectral_norm(nn.Linear(in_c * 2, out_c)), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Linear(out_c, out_c)),    nn.LeakyReLU(0.2))
    def forward(self, x, idx=None):
        if idx is None: idx = knn_graph(x, self.k)
        return self.mlp(get_edge_features(x, idx)).max(dim=2)[0]


class PointDiscriminator(nn.Module):
    def __init__(self, k=10):
        super().__init__()
        self.ec1 = EdgeConvSN(3,   64, k)
        self.ec2 = EdgeConvSN(64, 128, k)
        self.global_mlp = nn.Sequential(
            nn.utils.spectral_norm(nn.Linear(64+128, 256)), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Linear(256,    128)), nn.LeakyReLU(0.2),
            nn.utils.spectral_norm(nn.Linear(128,      1)))
    def forward(self, x):
        f1 = self.ec1(x)
        f2 = self.ec2(f1)
        g  = torch.cat([f1, f2], -1).max(dim=1)[0]
        return self.global_mlp(g)


class PointRefiner(nn.Module):
    def __init__(self, k=16):
        super().__init__()
        self.ec1 = EdgeConv(3,   64, k)
        self.ec2 = EdgeConv(64, 128, k)
        self.offset_head = nn.Sequential(
            nn.Linear(64+128+3, 128), nn.GELU(),
            nn.Linear(128, 64),       nn.GELU(),
            nn.Linear(64, 3))
        self.conf_head = nn.Sequential(
            nn.Linear(64+128+3, 64), nn.GELU(),
            nn.Linear(64, 1),        nn.Sigmoid())

    def forward(self, pts_local):
        f1 = self.ec1(pts_local)
        f2 = self.ec2(f1)
        feat = torch.cat([f1, f2, pts_local], -1)
        offset = 0.08 * torch.tanh(self.offset_head(feat))
        conf   = self.conf_head(feat).squeeze(-1)
        refined = torch.clamp(pts_local + offset, -1.0, 1.0)
        return refined, conf, offset


print("✓ PointRefiner, PointDiscriminator, EdgeConv, EdgeConvSN all defined.")

✓ PointRefiner, PointDiscriminator, EdgeConv, EdgeConvSN all defined.


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 10 — TEST SET EVALUATION
#   Uses GAN-refined output if available, otherwise generator-only.
#   Saves predicted VTKs under RESULTS_DIR and writes test_results.csv.
# ══════════════════════════════════════════════════════════════════════════════

USE_GAN_FOR_TEST = GAN_BEST_CKPT.exists()
print(f'Using GAN refiner for test: {USE_GAN_FOR_TEST}')

# Load generator
gen_eval = PPCNetV10().to(device)
gen_ckpt_for_test = GEN_BEST_CH_CKPT if GEN_BEST_CH_CKPT.exists() else GEN_BEST_CKPT
if not gen_ckpt_for_test.exists():
    gen_ckpt_for_test = GEN_CKPT_PATH
st_gen = torch.load(gen_ckpt_for_test, map_location=device, weights_only=False)
gen_eval.load_state_dict(st_gen['model'])
gen_eval.eval()
print(f'  Generator: epoch {st_gen["epoch"]+1} ({gen_ckpt_for_test.name})')

# Load refiner if available
ref_eval = None
if USE_GAN_FOR_TEST:
    ref_eval = PointRefiner(k=16).to(device)
    st_gan = torch.load(GAN_BEST_CKPT, map_location=device, weights_only=False)
    ref_eval.load_state_dict(st_gan['refiner'])
    ref_eval.eval()
    print(f'  Refiner  : epoch {st_gan["epoch"]+1} (gan_best_chamfer.pth)')

test_ds = LumbarDatasetV10(test_ids, augment=False)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         collate_fn=collate_keep_strings)

all_metrics = []
with torch.no_grad():
    for batch in test_loader:
        batch = move_to_device(batch)
        v_out = gen_eval(batch['drr_ap'], batch['drr_lp'],
                         batch['P_ap'], batch['P_lp'],
                         batch['center'], batch['scale'])

        if ref_eval is not None:
            refined, conf, _ = ref_eval(v_out['pred_local'])
            ref_world = local_to_world(refined, batch['center'], batch['scale'])
            pred = ref_world.cpu().numpy()
            conf_np = conf.cpu().numpy()
        else:
            pred = v_out['pred_world'].cpu().numpy()
            conf_np = None

        gt   = batch['gt_ppc_world'].cpu().numpy()
        pids = batch['patient_id']

        for b in range(pred.shape[0]):
            pts = pred[b]
            if conf_np is not None:
                c = conf_np[b]
                mask = c > 0.3
                if mask.sum() > 100:
                    good = pts[mask]
                    n_need = N_POINTS - len(good)
                    if n_need > 0:
                        pts = np.concatenate(
                            [good, good[np.random.choice(len(good), n_need, replace=True)]], 0)
                    else:
                        pts = good[:N_POINTS]
            m = chamfer_metrics_np(pts, gt[b])
            m['patient_id'] = pids[b]
            all_metrics.append(m)
            suffix = '_pred_gan.vtk' if ref_eval is not None else '_pred.vtk'
            save_vtk_points(pts, RESULTS_DIR / f'{pids[b]}{suffix}')

print(f'\n{"="*60}')
print(f'TEST SET RESULTS ({len(all_metrics)} patients)')
print(f'{"="*60}')
keys   = ['chamfer_mm','fscore_1mm','fscore_2mm','fscore_5mm','hausdorff_95']
labels = ['Chamfer (mm)','F@1mm','F@2mm','F@5mm','HD95 (mm)']
for k, lbl in zip(keys, labels):
    vals = [m[k] for m in all_metrics]
    print(f'  {lbl:<16} mean={np.mean(vals):.3f}  std={np.std(vals):.3f}  '
          f'median={np.median(vals):.3f}')

import csv
csv_path = RESULTS_DIR / 'test_results.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['patient_id'] + keys)
    writer.writeheader()
    writer.writerows(all_metrics)
print(f'Results saved to {csv_path}')


Using GAN refiner for test: True
  Generator: epoch 81 (gen_best_chamfer.pth)
  Refiner  : epoch 29 (gan_best_chamfer.pth)

TEST SET RESULTS (105 patients)
  Chamfer (mm)     mean=2.452  std=1.531  median=2.206
  F@1mm            mean=0.115  std=0.050  median=0.107
  F@2mm            mean=0.512  std=0.136  median=0.507
  F@5mm            mean=0.942  std=0.096  median=0.968
  HD95 (mm)        mean=6.506  std=7.141  median=5.159
Results saved to /apps/app/see_all_ai/ppc_network_v10/results/test_results.csv


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 11 — INFERENCE FUNCTION
#   Loads the best generator (and refiner if present) and predicts a PPC
#   from a single pair of DRR images + their projection matrices.
# ══════════════════════════════════════════════════════════════════════════════

def predict_ppc(drr_ap_path, drr_lp_path, p_ap_path, p_lp_path,
                output_path, gen_ckpt_path=None, gan_ckpt_path=None,
                center_mm=None, scale_mm=None, conf_threshold=0.3,
                use_gan=True):
    """Full v10 inference. If use_gan=True and a GAN checkpoint exists,
    runs generator → refiner → confidence-filtered output."""
    # Resolve checkpoints
    if gen_ckpt_path is None:
        gen_ckpt_path = GEN_BEST_CH_CKPT if GEN_BEST_CH_CKPT.exists() else GEN_BEST_CKPT
    if not Path(gen_ckpt_path).exists():
        gen_ckpt_path = GEN_CKPT_PATH
    if gan_ckpt_path is None:
        gan_ckpt_path = GAN_BEST_CKPT

    # Load generator
    net = PPCNetV10().to(device)
    s_g = torch.load(gen_ckpt_path, map_location=device, weights_only=False)
    net.load_state_dict(s_g['model'])
    net.eval()
    print(f'Loaded generator (epoch {s_g["epoch"]+1})')

    # Load refiner if requested
    use_refiner = use_gan and Path(gan_ckpt_path).exists()
    refiner = None
    if use_refiner:
        refiner = PointRefiner(k=16).to(device)
        s_r = torch.load(gan_ckpt_path, map_location=device, weights_only=False)
        refiner.load_state_dict(s_r['refiner'])
        refiner.eval()
        print(f'Loaded refiner (epoch {s_r["epoch"]+1})')

    img_norm = transforms.Normalize(mean=[0.485], std=[0.229])
    def _load(path):
        img = load_drr_image(path)
        return img_norm(torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float()).to(device)

    drr_ap_t = _load(drr_ap_path)
    drr_lp_t = _load(drr_lp_path)
    P_ap_t = torch.from_numpy(load_projection_matrix(p_ap_path)).unsqueeze(0).float().to(device)
    P_lp_t = torch.from_numpy(load_projection_matrix(p_lp_path)).unsqueeze(0).float().to(device)

    if center_mm is None: center_mm = [0.0, 0.0, 0.0]
    if scale_mm  is None: scale_mm  = [80.0, 80.0, 130.0]
    center_t = torch.tensor([center_mm], dtype=torch.float32, device=device)
    scale_t  = torch.tensor([scale_mm],  dtype=torch.float32, device=device)

    with torch.no_grad():
        out = net(drr_ap_t, drr_lp_t, P_ap_t, P_lp_t, center_t, scale_t)
        if use_refiner:
            refined, conf, _ = refiner(out['pred_local'])
            ref_world = local_to_world(refined, center_t, scale_t).squeeze(0).cpu().numpy()
            c = conf.squeeze(0).cpu().numpy()
            mask = c > conf_threshold
            pts = ref_world[mask] if mask.sum() > 100 else ref_world
        else:
            pts = out['pred_world'].squeeze(0).cpu().numpy()

    save_vtk_points(pts, output_path)
    print(f'Saved {len(pts)} points → {output_path}')
    return pts


print('predict_ppc() ready. Example:')
print("  pts = predict_ppc('drr_AP_0.png','drr_LP_90.png','P_AP_0.txt','P_LP_90.txt','out.vtk')")


predict_ppc() ready. Example:
  pts = predict_ppc('drr_AP_0.png','drr_LP_90.png','P_AP_0.txt','P_LP_90.txt','out.vtk')
